In [5]:
import pandas as pd

df = pd.read_csv("./../data/corrected_phishing_dataset.csv")

print("Liczba rekordów:", df.shape[0])
print("Liczba kolumn:", df.shape[1])

print("\nInformacje o kolumnach:")
df.info()
TARGET = "type"

Liczba rekordów: 651191
Liczba kolumn: 2

Informacje o kolumnach:
<class 'pandas.DataFrame'>
RangeIndex: 651191 entries, 0 to 651190
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   url     651191 non-null  str  
 1   type    651119 non-null  str  
dtypes: str(2)
memory usage: 9.9 MB


In [6]:
print("\nLiczba brakujących wartości w każdej kolumnie:")
missing = df.isna().sum()
missing = missing[missing>0]
print(missing.to_string())


Liczba brakujących wartości w każdej kolumnie:
type    72


In [7]:
print("Przed:", len(df))

df = df.dropna(subset=[TARGET]).reset_index(drop=True)

print("Po:", len(df))
print("Brakujące etykiety:", df[TARGET].isna().sum())

Przed: 651191
Po: 651119
Brakujące etykiety: 0


In [8]:
print("Duplikaty URL:", df.duplicated(subset=["url"]).sum())
conflicting_labels = (
    df.groupby("url")[TARGET]
    .nunique()
)

print(
    "URL z różnymi etykietami:",
    (conflicting_labels > 1).sum()
)

Duplikaty URL: 10072
URL z różnymi etykietami: 0


In [9]:
before = len(df)

df = df.drop_duplicates().reset_index(drop=True)

print("Usunięte duplikaty:", before - len(df))
print("Pozostałe rekordy:", len(df))

Usunięte duplikaty: 10072
Pozostałe rekordy: 641047


In [10]:
df[TARGET].value_counts()

type
benign        428182
defacement     95308
phishing       93912
malware        23645
Name: count, dtype: int64

In [11]:
import re

urls = df["url"]

http_count = urls.str.match(
    r"^http://", flags=re.IGNORECASE, na=False
)

https_count = urls.str.match(
    r"^https://", flags=re.IGNORECASE, na=False
)

no_protocol_count = len(df) - http_count.sum() - https_count.sum()

print(f"HTTP: {http_count.sum()}")
print(f"HTTPS: {https_count.sum()}")
print(f"Bez HTTP/HTTPS: {no_protocol_count}")

HTTP: 162317
HTTPS: 15637
Bez HTTP/HTTPS: 463093


In [17]:
import re
from urllib.parse import urlsplit

SCHEME_PATTERN = re.compile(r"^[A-Za-z][A-Za-z0-9+.-]*://")


def is_valid_url(url):
    if not isinstance(url, str) or not url.strip():
        return False

    try:
        # Parsowanie analogiczne do parse_url()
        if SCHEME_PATTERN.match(url) or url.startswith("//"):
            parsed = urlsplit(url)
        else:
            parsed = urlsplit("//" + url)

        # Sprawdzenie, czy udało się wyodrębnić host
        hostname = parsed.hostname

        if not hostname:
            return False

        # Odrzucenie URL-i zawierających znaki sterujące
        if re.search(r"[\x00-\x1F\x7F-\x9F]", url):
            return False

        return True

    except ValueError:
        return False


# Sprawdzenie rekordów
valid_mask = df["url"].apply(is_valid_url)

# Wyodrębnienie nieprawidłowych rekordów
df_invalid = df.loc[~valid_mask].copy()

print("Liczba wszystkich rekordów:", len(df))
print("Liczba nieprawidłowych URL-i:", len(df_invalid))

# Wyświetlenie nieprawidłowych URL-i wraz z etykietami
display(df_invalid[["url", "type"]])
df = df.loc[valid_mask].copy().reset_index(drop=True)
print("Rekordy po filtrowaniu:", len(df))

Liczba wszystkich rekordów: 641047
Liczba nieprawidłowych URL-i: 34


,url,type
61370,http://www.scsrxx.com/js/?https://secure.runes...,phishing
520493,http://gkjx168.com/images?us.battle.net/login/...,phishing
523151,http://email302.com/l/5fc15ea15e66c082e33c48ba...,phishing
523240,http://email302.com/l/5fc15ea15e66c082e33c48ba...,phishing
523243,http://email302.com/l/5fc15ea15e66c082e33c48ba...,phishing
523244,http://email302.com/l/5fc15ea15e66c082e33c48ba...,phishing
523245,http://email302.com/l/5fc15ea15e66c082e33c48ba...,phishing
523255,http://email302.com/l/5fc15ea15e66c082e33c48ba...,phishing
523256,http://email302.com/l/5fc15ea15e66c082e33c48ba...,phishing
523259,http://email302.com/l/5fc15ea15e66c082e33c48ba...,phishing


Rekordy po filtrowaniu: 641013


In [18]:
df.to_csv('./../data/deduplicated_corrected_phishing_dataset.csv',index=False, encoding="utf-8")